## Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

## Load Libraries

In [3]:
library(tidyr)
library(dplyr)
library(ggplot2)
library(tibble)
library(patchwork)
library(ggpubr)
library(stringr)

# Load Metadata

In [4]:
# Read in metadata
metadata <- read.csv("metadata.csv")

# Split metadata by gene to create named lists
metadata <- split(metadata, metadata$Gene)

# Remove any empty data frames that might result from filtering
metadata <- metadata[sapply(metadata, nrow) > 0]

# View
str(metadata)

List of 1
 $ MECP2:'data.frame':	32 obs. of  23 variables:
  ..$ Gene                      : chr [1:32] "MECP2" "MECP2" "MECP2" "MECP2" ...
  ..$ Sample_ID                 : chr [1:32] "KB1" "KB2" "KB3" "KB19" ...
  ..$ Sample_Name               : chr [1:32] "CO_R270X_B1_d18_A1" "CO_R270X_B1_d18_A2" "CO_R270X_B1_d18_A3" "CO_R270X_B2_d18_A1" ...
  ..$ Cell_Type                 : chr [1:32] "Cerebral_Organoid" "Cerebral_Organoid" "Cerebral_Organoid" "Cerebral_Organoid" ...
  ..$ Genotype                  : chr [1:32] "MUT" "MUT" "MUT" "MUT" ...
  ..$ Time_Point                : chr [1:32] "D18" "D18" "D18" "D18" ...
  ..$ Group                     : chr [1:32] "R270X_D18" "R270X_D18" "R270X_D18" "R270X_D18" ...
  ..$ Replicate                 : int [1:32] 1 2 3 4 5 6 1 2 3 4 ...
  ..$ Clean_Sample_Name         : chr [1:32] "R270X_D18_1" "R270X_D18_2" "R270X_D18_3" "R270X_D18_4" ...
  ..$ Batch                     : int [1:32] 0 0 0 1 1 1 0 0 0 1 ...
  ..$ D18_MUT_vs_D18_WT         : int 

In [5]:
# View groups 
for (gene_name in names(metadata)) {
  cat("\n=== Group counts for", gene_name, "===\n")
  print(table(metadata[[gene_name]]$Group))
}


=== Group counts for MECP2 ===

R270X_D18 R270X_D46 R270X_D90    WT_D18    WT_D46    WT_D90 
        6         4         6         6         5         5 


# Load Counts

In [6]:
# Read in counts
counts <- read.delim("05_gene_counts/MECP2_normalized_counts.txt")

# Read in metadata and select only the mapping columns to rename samples
mapping_df <- read.csv("metadata.csv")[, c("Sample_Name", "Clean_Sample_Name")]

# Create mapping
name_mapping <- setNames(mapping_df$Clean_Sample_Name, mapping_df$Sample_Name)

# Remove whitespace
colnames(counts) <- trimws(colnames(counts))

# Rename columns
colnames(counts) <- name_mapping[colnames(counts)]

# Drop columns that don't map
counts <- counts[, !is.na(colnames(counts))]

# View
head(counts)

,R270X_D18_1,R270X_D18_2,R270X_D18_3,R270X_D18_4,R270X_D18_5,R270X_D18_6,R270X_D46_1,R270X_D46_2,R270X_D46_3,R270X_D46_4,⋯,WT_D46_1,WT_D46_2,WT_D46_3,WT_D46_4,WT_D46_5,WT_D90_1,WT_D90_2,WT_D90_3,WT_D90_4,WT_D90_5
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A0A024R0Q4,1.5955131,1.776155,1.051596,1.325690,1.876562,1.2972164,1.441147,1.793023,1.9981165,1.997648,⋯,2.0175780,2.258242,1.785696,1.764698,0.45101005,1.779600,2.520144,1.991368,2.835845,1.990374
A0A024R5G2,6.5338210,6.600780,6.693552,6.926624,6.770172,7.3023333,6.433098,6.170686,6.3606658,6.624274,⋯,6.2838073,6.174269,6.288047,6.365530,7.34696247,5.865562,5.584527,6.156353,5.753729,5.910733
A0A024RAA5,1.0213060,1.776155,1.051596,2.429266,2.176069,0.6618769,2.887948,2.543490,3.4020222,2.988700,⋯,3.3312998,3.147539,3.043483,2.958170,2.22289674,2.911341,3.428417,2.981747,2.097798,1.990374
A0A024RAJ1,0.4523278,1.148576,1.271631,2.222271,1.263736,2.7728478,1.441147,0.917640,1.7066873,1.232729,⋯,0.7735121,1.685210,1.922383,0.060944,-0.50075523,2.622903,2.520144,1.867580,1.095090,1.009635
A0A024RBW1,4.0801219,4.367480,4.006765,4.252824,3.325689,2.9467234,1.032681,1.618853,0.8502006,1.873739,⋯,1.0311523,1.537162,1.274966,2.524291,0.05223064,2.361044,1.252954,1.732164,2.097798,1.415074
A0A024RDL5,3.2182169,2.878073,3.360271,3.382059,3.699480,3.8790375,4.401071,4.862035,4.1892690,4.305232,⋯,4.0945736,4.635590,4.011270,3.955019,4.61330215,3.095188,3.184200,2.920766,4.129587,4.880361


# Subset to Genes of Interest

In [7]:
# Load genes of interest
genes_of_interest <- read.csv("go_bp_2025_apoptotic_markers.csv")

genes_of_interest <- genes_of_interest %>%
  mutate(Gene_List = strsplit(Gene_List, "|", fixed = TRUE)) %>%
  deframe()

# View
str(genes_of_interest)

List of 2
 $ Negative Regulation of Execution Phase of Apoptosis (GO:1900118): chr [1:5] "CIDEA" "CXCR3" "DFFA" "FZD9" ...
 $ Positive Regulation of Execution Phase of Apoptosis (GO:1900119): chr [1:11] "CXCR3" "DLC1" "GLMN" "HSPD1" ...


In [8]:
# Select only genes of interest
all_target_genes <- unlist(genes_of_interest, use.names = FALSE)

counts <- counts[rownames(counts) %in% all_target_genes, ]

# View
head(counts)

,R270X_D18_1,R270X_D18_2,R270X_D18_3,R270X_D18_4,R270X_D18_5,R270X_D18_6,R270X_D46_1,R270X_D46_2,R270X_D46_3,R270X_D46_4,⋯,WT_D46_1,WT_D46_2,WT_D46_3,WT_D46_4,WT_D46_5,WT_D90_1,WT_D90_2,WT_D90_3,WT_D90_4,WT_D90_5
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
GLMN,2.416111,2.010496,1.781950,1.842537,0.6359416,2.073254,2.240008,0.5800753,1.535439,2.654025,⋯,1.249679,1.819470,1.922383,1.256282,1.237189,1.269539,0.4619031,2.211018,-0.5007552,0.04740308
HSPD1,4.945112,5.122572,4.990282,4.139532,3.6516540,2.677363,4.613268,4.4325444,4.037056,4.936221,⋯,4.724783,4.441002,4.915218,4.856253,5.003112,4.281841,4.8347871,5.075862,4.4218889,4.29670594
HTRA2,5.263514,5.342568,4.974697,5.502761,5.1982337,5.431749,5.242846,5.1289956,4.955583,5.515233,⋯,5.035307,5.098724,5.288443,5.568881,5.368515,5.202960,5.1529548,5.117218,5.2017244,5.72719495
SIRT2,1.595513,1.642969,1.051596,1.980522,0.9837208,2.073254,2.338786,2.5434898,2.124368,2.217490,⋯,3.069362,2.517278,2.162134,2.344853,3.163497,3.036470,2.9538715,2.981747,2.9063121,1.99037414
TP53,4.289816,4.063071,4.037094,3.984131,4.5595860,3.879038,2.431232,3.5042784,3.244104,3.209809,⋯,2.597987,2.594226,3.102226,2.437360,4.237556,1.269539,1.8970585,2.105369,2.5135118,2.48716013


# Make Expression Box Plot

In [9]:
# Reshape data 
counts_long <- counts %>%
  rownames_to_column(var = "gene") %>%
  pivot_longer(
    cols = -gene,         
    names_to = "sample",   
    values_to = "expression"
  ) %>%
  mutate(
    group = gsub("_\\d+$", "", sample),
    date = gsub(".*_", "", group)
  )

# View
head(counts_long)

gene,sample,expression,group,date
<chr>,<chr>,<dbl>,<chr>,<chr>
GLMN,R270X_D18_1,2.4161108,R270X_D18,D18
GLMN,R270X_D18_2,2.0104956,R270X_D18,D18
GLMN,R270X_D18_3,1.7819503,R270X_D18,D18
GLMN,R270X_D18_4,1.8425369,R270X_D18,D18
GLMN,R270X_D18_5,0.6359416,R270X_D18,D18
GLMN,R270X_D18_6,2.0732541,R270X_D18,D18


In [10]:
# Convert date to factor with desired order
counts_long$date <- factor(counts_long$date, levels = c("D18", "D46", "D90"))

# Sort by date, then genotype
counts_long$genotype <- gsub("_.*", "", counts_long$group)

# Create ordered factor
counts_long$group_ordered <- factor(counts_long$group, 
                                     levels = c("WT_D18", "R270X_D18", 
                                                "WT_D46", "R270X_D46",
                                                "WT_D90", "R270X_D90"))

# Replace underscores with spaces for plot labeling
levels(counts_long$group_ordered) <- gsub("_", " ", levels(counts_long$group_ordered))

# Replace R270X with MUT
levels(counts_long$group_ordered) <- gsub("R270X", "MUT", levels(counts_long$group_ordered))

In [11]:
# Create a data frame mapping each gene to its cell type
gene_celltype_map <- data.frame(
  gene = unlist(genes_of_interest, use.names = FALSE),
  cell_type = rep(names(genes_of_interest), lengths(genes_of_interest))
)

# Merge the mapping with long counts data
counts_long_mapped <- counts_long %>%
  left_join(gene_celltype_map, by = "gene") %>%
  filter(!is.na(cell_type)) 

Warning message in left_join(., gene_celltype_map, by = "gene"):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 33 of `x` matches multiple rows in `y`.
ℹ Row 8 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship = "many-to-many"` to silence this warning.”


In [12]:
# Remove NA
counts_long_clean <- counts_long_mapped %>%
  filter(!is.na(expression))

# Max genes per row
max_cols <- 5 

In [14]:
# List to store the final combined plot for each row
row_plots <- list() 

# Loop through each cell type to build one complete row at a time
for (ct in names(genes_of_interest)) {
  
  detected_genes <- genes_of_interest[[ct]][genes_of_interest[[ct]] %in% counts_long_clean$gene]
  if (length(detected_genes) == 0) next
  
  # Create the rotated label plot for the left side
  label_p <- ggplot() +
    annotate("text", x = 0.5, y = 0.5, label = str_wrap(ct, width = 30), angle = 90,
             size = 4.5, fontface = "bold", color = "black") +
    theme_void() +
    theme(plot.margin = margin(0, -15, 0, 0))
  
  # Create the gene plots for the row
  gene_plots <- list()
  for (gene_name in detected_genes) {
    plot_data <- counts_long_clean %>% filter(gene == gene_name)
    
    p <- ggplot(plot_data, aes(x = group_ordered, y = expression, fill = date)) +
      geom_boxplot(outlier.shape = NA) +
      geom_jitter(width = 0, alpha = 1, size = 0.8) +
      theme_minimal() +
      theme(
        axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
        plot.title = element_text(hjust = 0.5, face = "bold", size = 12),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        axis.line = element_line(color = "black"),
        legend.position = "none"
      ) +
      labs(title = gene_name, 
           x = "Sample Group", 
           y = "Normalized Read Count", 
           fill = "Time Point") +
      scale_fill_manual(values = c("D18" = "aliceblue", 
                                   "D46" = "lightblue", 
                                   "D90" = "steelblue"))
    
    gene_plots[[length(gene_plots) + 1]] <- p
  }
  
  # Pad with empty plots to ensure every row has exactly max_cols gene plots
  n_empty <- max_cols - length(detected_genes)
  empty_plot <- ggplot() + theme_void() + theme(plot.margin = margin(0, 0, 0, 0))
  if (n_empty > 0) {
    for (k in 1:n_empty) {
      gene_plots[[length(gene_plots) + 1]] <- empty_plot
    }
  }
  
  # Combine the label and the gene plots into a single row
  all_row_plots <- c(list(label_p), gene_plots)
  row_p <- wrap_plots(all_row_plots, ncol = 1 + max_cols, widths = c(1, rep(1.5, max_cols)))
  
  row_plots[[length(row_plots) + 1]] <- row_p
}

# Stack all the rows
final_plot <- wrap_plots(row_plots, ncol = 1, guides = "collect") &
  theme(legend.position = "bottom")

# Save
ggsave("apoptotic_marker_expression_plot.pdf",
  plot = final_plot,
  device = "pdf",
  path = "07_DEG_Figures",
  scale = 1,
  width = 15,
  height = 8,
  units = "in",
  dpi = 600,
  create.dir = TRUE,
)

# Make Expression Box Plots with Significance Annotated

In [15]:
# Define comparisons of interest
comparisons <- list(
  # WT vs MUT per time point
  c("WT D18", "MUT D18"),
  c("WT D46", "MUT D46"),
  c("WT D90", "MUT D90"),
  
  # WT across time points
  c("WT D18", "WT D46"),
  c("WT D18", "WT D90"),
  c("WT D46", "WT D90"),
  
  # MUT across time points
  c("MUT D18", "MUT D46"),
  c("MUT D18", "MUT D90"),
  c("MUT D46", "MUT D90")
)

In [16]:
# Remove NA
counts_long_clean <- counts_long_mapped %>%
  filter(!is.na(expression))

# Max genes per row
max_cols <- 5 

In [17]:
# List to store the final combined plot for each row
row_plots <- list() 

# Loop through each cell type to build one complete row at a time
for (ct in names(genes_of_interest)) {
  
  detected_genes <- genes_of_interest[[ct]][genes_of_interest[[ct]] %in% counts_long_clean$gene]
  if (length(detected_genes) == 0) next
  
  # Create the rotated label plot for the left side
  label_p <- ggplot() +
    annotate("text", x = 0.5, y = 0.5, label = str_wrap(ct, width = 30), angle = 90,
             size = 4.5, fontface = "bold", color = "black") +
    theme_void() +
    theme(plot.margin = margin(0, -15, 0, 0))
  
  # Create the gene plots for the row
  gene_plots <- list()
  for (gene_name in detected_genes) {
    plot_data <- counts_long_clean %>% filter(gene == gene_name)
    
    p <- ggplot(plot_data, aes(x = group_ordered, y = expression, fill = date)) +
      geom_boxplot(outlier.shape = NA) +
      geom_jitter(width = 0, alpha = 1, size = 0.8) +

      # Significance testing
      geom_pwc(method = "wilcox.test",
               p.adjust.method = "BH",
               label = "p.signif",
               step.increase = 0.1,
               size = 0.2,
               label.size = 2,
               hide.ns = TRUE) +     
      
      theme_minimal() +
      theme(
        axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
        plot.title = element_text(hjust = 0.5, face = "bold", size = 12),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        axis.line = element_line(color = "black"),
        legend.position = "none"
      ) +
      labs(title = gene_name, 
           x = "Sample Group", 
           y = "Normalized Read Count", 
           fill = "Time Point") +
      scale_fill_manual(values = c("D18" = "aliceblue", 
                                   "D46" = "lightblue", 
                                   "D90" = "steelblue"))
    
    gene_plots[[length(gene_plots) + 1]] <- p
  }
  
  # Pad with empty plots 
  n_empty <- max_cols - length(detected_genes)
  empty_plot <- ggplot() + theme_void() + theme(plot.margin = margin(0, 0, 0, 0))
  if (n_empty > 0) {
    for (k in 1:n_empty) {
      gene_plots[[length(gene_plots) + 1]] <- empty_plot
    }
  }
  
  # Combine the label and the gene plots into a single row
  all_row_plots <- c(list(label_p), gene_plots)
  row_p <- wrap_plots(all_row_plots, ncol = 1 + max_cols, widths = c(1, rep(1.5, max_cols)))
  
  row_plots[[length(row_plots) + 1]] <- row_p
}

# Stack all the rows
final_plot <- wrap_plots(row_plots, ncol = 1, guides = "collect") &
  theme(legend.position = "bottom")

# Save
ggsave("apoptic_marker_expression_plot_significance.pdf",
  plot = final_plot,
  device = "pdf",
  path = "07_DEG_Figures",
  scale = 1,
  width = 15,
  height = 9,
  units = "in",
  dpi = 600,
  create.dir = TRUE,
)